# Exercise 3: Coverage comparison

*Finite-State Morphology · version 0.1.0*

> Run the TurkicNLP Apertium analyser on 20 words of your choice in Turkish and in Altai. Count how many words receive at least one analysis vs. how many return empty (with HFST, `analyser.lookup(word)` returns an empty tuple; in a `morph` pipeline, unknown words get `upos == "X"`). Compare the results. What types of words does the Altai analyser fail on most often? Does the neural analyser (`morph_neural`) fill these gaps correctly?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. These notebooks compile lexc and twolc grammars, which needs the compiled `hfst` package; on PyPI it had wheels for Python 3.11 but not for 3.13 when checked on 19 October 2026, so use Python 3.11 or 3.12 if `pip` cannot find it. The Apertium transducers are downloaded once with `turkicnlp.download`.

To make the two languages comparable, the worked example takes the words from the same sentences: the parallel challenge set of the TurkicNLP paper (its Appendix C), which the chapter's evaluation table is based on. The first 20 distinct words other than the name *Ali* are looked up in each transducer, first in their written form and then lowercased. The last cell runs the neural analyser on the Altai sentences; it needs the Glot500 model, which is downloaded from Hugging Face on first use.

In [ ]:
import tempfile
from pathlib import Path
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

def compile_grammar(lexc_text, twolc_text=None):
    """Compile a lexc lexicon (and optional twolc rules); return (analyser, generator) in lookup format."""
    work = Path(tempfile.mkdtemp())
    (work / "g.lexc").write_text(lexc_text, encoding="utf-8")
    gen = hfst.compile_lexc_file(str(work / "g.lexc"))
    if twolc_text:
        (work / "g.twolc").write_text(twolc_text, encoding="utf-8")
        hfst.compile_twolc_file(str(work / "g.twolc"), str(work / "g.twol.hfst"))
        stream, rules = hfst.HfstInputStream(str(work / "g.twol.hfst")), []
        while not stream.is_eof():
            rules.append(stream.read())
        stream.close()
        gen.compose_intersect(tuple(rules))
    gen.minimize()
    ana = hfst.HfstTransducer(gen)
    ana.invert()
    ana.minimize()
    for t in (ana, gen):
        t.convert(hfst.ImplementationType.HFST_OLW_TYPE)
    return ana, gen

def apertium(lang, script, kind="automorf"):
    """An Apertium transducer distributed with TurkicNLP (downloaded on first use)."""
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.{kind}.hfst"
    stream = hfst.HfstInputStream(str(path))
    fst = stream.read()
    stream.close()
    return fst

lookup = lambda fst, s: sorted({out for out, _ in fst.lookup(s)})

In [ ]:
import re
from collections import Counter

# The parallel challenge sentences of the TurkicNLP paper (Appendix C), Turkish and Altai.
TUR = ["Ali'nin kitabı masada.", "Ali kitabı gördü.", "Ali okulda bekliyor.", "Ali okuldan çıktı.", "Ali okula girdi.",
       "Çocuklar parkta oynadı.", "Benim evim büyük.", "Ali dün geldi.", "Ali şimdi geliyor.", "Ali yarın gelecek.",
       "Ali okula gitmedi.", "Ali okula gitti mi?", "Ali, kapıyı aç!", "Ali gelse iyi olurdu.", "Ali gelmiş.",
       "Kapı açıldı.", "Onlar birbirini gördü.", "Biz seni tanıyoruz.", "Üç öğrenci geldi.", "Ali gülerek konuştu."]
ALT = ["Алинин бичиги столдо.", "Али бичикти кӧрди.", "Али школдо сакып јат.", "Али школдоҥ чыкты.", "Али школго кирди.",
       "Балдар паркта ойнодылар.", "Мениҥ айылым јаан.", "Али тӱӱне келди.", "Али эмди келип јат.", "Али эртен келер.",
       "Али школго барган јок.", "Али школго барды ба?", "Али, эжикти ач!", "Али келзе, јараар эди.", "Али келип калган.",
       "Эжик ачылды.", "Олор бой-бойын кӧрдилӧр.", "Бис сени билерис.", "Ӱч студент келди.", "Али кӱлӱп куучындады."]

def words(sentences):
    """Distinct word types other than the name Ali, in order of first occurrence."""
    seen = []
    for s in sentences:
        for w in re.findall(r"[^\W\d_]+(?:['’-][^\W\d_]+)*", s):
            if w.lower() not in {"ali", "али"} and w.lower() not in [x.lower() for x in seen]:
                seen.append(w)
    return seen[:20]

def known(fst, w):
    return bool(fst.lookup(w) or fst.lookup(w.lower()))

for lang, script, sentences in [("tur", "Latn", TUR), ("alt", "Cyrl", ALT)]:
    fst = apertium(lang, script)
    ws = words(sentences)
    unknown = [w for w in ws if not known(fst, w)]
    print(f"{lang}: {len(ws) - len(unknown)}/{len(ws)} words analysed; unknown: {unknown}")

In [ ]:
try:
    from turkicnlp import Pipeline
    turkicnlp.download("alt", processors=["tokenize", "morph_neural"])
    neural = Pipeline("alt", processors=["tokenize", "morph_neural"], tokenize_backend="rule")
    for s in ALT:
        for w in neural(s).words:
            print(f"{w.text:<14} {w.lemma or '_':<10} {w.upos or '_':<6} {w.feats or '_'}")
except Exception as exc:
    print("Neural analyser not available here:", type(exc).__name__, str(exc)[:120])

## Worked solution

Output of `experiments/ch06_exercises/run_ch06_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
tur: 20/20 words analysed; unknown: []
alt: 7/20 words analysed; unknown: ['Алинин', 'бичиги', 'столдо', 'бичикти', 'школдо', 'сакып', 'школдоҥ', 'школго', 'Балдар', 'паркта', 'ойнодылар', 'айылым', 'тӱӱне']
```

```
  ↓ Downloading cis-lmu/glot500-base backbone...
Neural analyser not available here: ProxyError 403 Forbidden
```

On the same 20 sentences, the Turkish transducer analyses all 20 words and the Altai transducer 7 of 20. The 13 Altai failures fall into three groups:

- the name with a case suffix (*Алинин* 'Ali's');
- loanwords with case suffixes (*школдо*, *школдоҥ*, *школго* 'school', *столдо* 'table', *паркта* 'park');
- everyday native vocabulary: *бичиги* and *бичикти* (two case forms of 'book'), *балдар* 'children', *ойнодылар* 'played', *айылым* 'my house', *тӱӱне* 'yesterday', *сакып* 'waiting'.

The failures are therefore lexical, and they include core words, not only loans. This is what a prototype transducer with a small lexicon looks like, and it fits the low Altai detection rate (40%) in the chapter's evaluation table. The neural analyser could not be run in the environment used for this solution, because the Glot500 model could not be downloaded; the cell prints that instead of a result. Twenty word types from one parallel set are a diagnostic, not a coverage estimate for Altai text.

## Your turn — extension

1. Run the neural cell where the Glot500 model can be downloaded, and check each of its analyses of the unknown Altai words by hand: lemma, part of speech and features.
2. Repeat with Khakas (`kjh`), the other prototype transducer with a low score in the chapter's table.
3. Count *tokens* instead of distinct words on a longer Altai text. How does the coverage figure change?